# 00 · Viabilidad y fuentes

**Etapa 0.** Objetivo: comprobar con los datos reales si el proyecto es viable antes de plantear el análisis.

Pregunta de fondo: ¿existen, en los **mismos estudiantes argentinos**, el rendimiento, el nivel socioeconómico, el uso de IA y los pesos muestrales de PISA?

Este notebook solo audita. No contiene resultados sobre las hipótesis.


## 1. Preparación

Se localiza la raíz del proyecto subiendo carpetas hasta encontrar `README.md`, para no depender de rutas de ninguna computadora. Los datos viven en `data/raw/`, que no se sube a GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import pyreadstat

raiz = Path.cwd()
while not (raiz / "README.md").exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("No se encontró README.md: ejecutá el notebook dentro del proyecto.")
    raiz = raiz.parent

ruta_2025 = raiz / "data" / "raw" / "pisa2025" / "extraido" / "CY09_MS_STU_PUF.sav"
ruta_2022 = raiz / "data" / "raw" / "pisa2022" / "sav" / "CY08MSP_STU_QQQ.SAV"
ruta_codebook = raiz / "data" / "raw" / "pisa2025" / "docs" / "PISA2025_Codebook.xlsx"

print("Datos PISA 2025 presentes:", ruta_2025.exists())
print("Datos PISA 2022 presentes:", ruta_2022.exists())
print("Codebook presente:", ruta_codebook.exists())
print("pandas", pd.__version__, "| pyreadstat", pyreadstat.__version__)

Datos PISA 2025 presentes: True
Datos PISA 2022 presentes: True
Codebook presente: True
pandas 3.1.1 | pyreadstat 1.3.7


## 2. Matriz de fuentes

Estado de cada fuente según lo verificado hasta ahora. "No verificado" significa que no pudimos confirmarlo con una página oficial.

In [2]:
fuentes = pd.DataFrame([
    {"fuente": "OCDE PISA 2025, estudiantes", "rol": "Principal (Plan A)", "estado": "descargado y auditado (secciones 3 a 5)",
     "uso de IA": "ST438Q01DA a ST438Q04DA, IC170Q10DA, IC171Q10DA, AIUSESCH", "rendimiento": "Valores plausibles de matemática, lectura y ciencias",
     "pesos": "W_FSTUWT y 80 pesos replicados", "terminos de uso": "no verificado"},
    {"fuente": "OCDE PISA 2022, estudiantes", "rol": "Respaldo / contexto", "estado": "descargado",
     "uso de IA": "Se verifica en la sección 6", "rendimiento": "Se verifica en la sección 6",
     "pesos": "Se verifica en la sección 6", "terminos de uso": "no verificado"},
    {"fuente": "UNICEF Kids Online Argentina 2025", "rol": "Contexto descriptivo", "estado": "descargado (informe PDF)",
     "uso de IA": "Según el informe, solo resultados agregados (no verificado en este notebook)", "rendimiento": "No",
     "pesos": "No aplica", "terminos de uso": "Permiso escrito para reutilizar material"},
])
fuentes

,fuente,rol,estado,uso de IA,rendimiento,pesos,terminos de uso
0,"OCDE PISA 2025, estudiantes",Principal (Plan A),descargado y auditado (secciones 3 a 5),"ST438Q01DA a ST438Q04DA, IC170Q10DA, IC171Q10D...","Valores plausibles de matemática, lectura y ci...",W_FSTUWT y 80 pesos replicados,no verificado
1,"OCDE PISA 2022, estudiantes",Respaldo / contexto,descargado,Se verifica en la sección 6,Se verifica en la sección 6,Se verifica en la sección 6,no verificado
2,UNICEF Kids Online Argentina 2025,Contexto descriptivo,descargado (informe PDF),"Según el informe, solo resultados agregados (n...",No,No aplica,Permiso escrito para reutilizar material


## 3. Estructura del archivo de estudiantes de PISA 2025

Se leen primero solo los metadatos (nombres de columnas y número de filas), sin cargar los datos.

In [3]:
_, meta = pyreadstat.read_sav(ruta_2025, metadataonly=True)
columnas = meta.column_names

print("Filas:", meta.number_rows)
print("Columnas:", len(columnas))
cantidad_replicados = 0
for c in columnas:
    if c.startswith("W_FSTURWT"):
        cantidad_replicados = cantidad_replicados + 1
print("Pesos replicados:", cantidad_replicados)
valores_plausibles_mate = []
for c in columnas:
    if c.startswith("PV") and c.endswith("MATH"):
        valores_plausibles_mate.append(c)
print("Valores plausibles de matemática:", len(valores_plausibles_mate), "columnas, por ejemplo", valores_plausibles_mate[:3])

Filas: 755721
Columnas: 885
Pesos replicados: 80
Valores plausibles de matemática: 10 columnas, por ejemplo ['PV1MATH', 'PV2MATH', 'PV3MATH']


## 4. Variables clave en Argentina

Se cargan solo las columnas necesarias. Las preguntas de IA son de **frecuencia de uso de chatbots** para cuatro actividades escolares (resumir, investigar, redactar borradores, ayudar a aprender), con la escala 1 a 5 definida en el archivo (las etiquetas se leen de los metadatos en la sección 5).

El archivo trae un `NaN` para cualquier faltante; todavía no distinguimos entre filtro del cuestionario, no administrado y no respuesta.

In [4]:
vars_ia = ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA", "IC170Q10DA", "IC171Q10DA", "AIUSESCH"]
vars_resto = ["CNT", "CNTSCHID", "ESCS", "W_FSTUWT", "PV1MATH", "PV1READ", "PV1SCIE", "PV10MATH", "PV10READ", "PV10SCIE"]

datos, _ = pyreadstat.read_sav(ruta_2025, usecols=vars_resto + vars_ia)
argentina = datos.loc[datos["CNT"] == "ARG"].copy()

print("Estudiantes en el archivo:", len(datos), "| países:", datos["CNT"].nunique())
print("Estudiantes de Argentina:", len(argentina))
print("Escuelas de Argentina:", argentina["CNTSCHID"].nunique())

Estudiantes en el archivo: 755721 | países: 90
Estudiantes de Argentina: 11092
Escuelas de Argentina: 412


In [5]:
# Porcentaje de estudiantes de Argentina que tiene dato en cada variable
columnas_a_revisar = vars_resto[2:] + vars_ia
cobertura = argentina[columnas_a_revisar].notna().mean() * 100
cobertura = cobertura.round(1)
print(cobertura)

ESCS           88.4
W_FSTUWT      100.0
PV1MATH       100.0
PV1READ       100.0
PV1SCIE       100.0
PV10MATH      100.0
PV10READ      100.0
PV10SCIE      100.0
ST438Q01DA     70.8
ST438Q02DA     70.0
ST438Q03DA     69.2
ST438Q04DA     69.0
IC170Q10DA     50.8
IC171Q10DA     47.4
AIUSESCH       69.7
dtype: float64


**Lectura.** Rendimiento y peso final están completos. El ESCS tiene dato para casi 9 de cada 10 estudiantes. Las cuatro preguntas de IA tienen dato para alrededor de 7 de cada 10, y las de uso de IA en la escuela (`IC170Q10DA`) y fuera de ella (`IC171Q10DA`) para la mitad. Esto es un hallazgo de cobertura, no una conclusión: antes de analizar hay que entender el origen de esos faltantes.

## 5. Distribución cruda de las cuatro preguntas de IA

Solo conteos (sin pesos) para ver si hay variación utilizable. **No son estimaciones para la población argentina.** La inferencia poblacional exige los pesos y la replicación, que se tratan en etapas posteriores.

In [6]:
# Las etiquetas de la escala salen del propio archivo (vienen en inglés)
_, meta_etiquetas = pyreadstat.read_sav(ruta_2025, metadataonly=True)
etiquetas = meta_etiquetas.variable_value_labels["ST438Q01DA"]
print(etiquetas)

preguntas = {
    "ST438Q01DA": "Resumir un texto",
    "ST438Q02DA": "Investigar un tema nuevo",
    "ST438Q03DA": "Redactar borradores",
    "ST438Q04DA": "Ayudar a aprender",
}

# Una columna por pregunta, contando cuántos estudiantes eligieron cada respuesta
tabla = pd.DataFrame()
for codigo in preguntas:
    nombre = preguntas[codigo]
    tabla[nombre] = argentina[codigo].value_counts().sort_index()

# Cambiamos el número de respuesta por su etiqueta
tabla.index = [etiquetas[numero] for numero in tabla.index]

# Agregamos la fila de estudiantes sin dato
sin_dato = []
for codigo in preguntas:
    sin_dato.append(argentina[codigo].isna().sum())
tabla.loc["Sin dato (NaN)"] = sin_dato
print(tabla)

{1.0: 'Never or almost never', 2.0: 'About once or twice a year', 3.0: 'About once or twice a month', 4.0: 'About once or twice a week', 5.0: 'Every day or almost every day', 95.0: 'valid skip (filter rule)', 96.0: 'system missing', 97.0: 'not administered/not applicable', 98.0: 'invalid', 99.0: 'no response'}
                               Resumir un texto  Investigar un tema nuevo  \
Never or almost never                      1786                      1326   
About once or twice a year                 1163                      1254   
About once or twice a month                1621                      1902   
About once or twice a week                 1821                      1863   
Every day or almost every day              1465                      1421   
Sin dato (NaN)                             3236                      3326   

                               Redactar borradores  Ayudar a aprender  
Never or almost never                         2375               1176  
Abou

## 5b. ¿Los mismos estudiantes tienen todo?

Hasta acá se vio la cobertura de cada variable por separado. Lo que importa es cuántos estudiantes tienen **a la vez** rendimiento, ESCS, peso y las cuatro preguntas de IA (casos completos), y si los 80 pesos replicados tienen dato para Argentina.

In [7]:
# Estudiantes que tienen dato en todas las variables que vamos a necesitar
vars_completas = ["PV1MATH", "ESCS", "W_FSTUWT", "ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA"]
tienen_todo = argentina[vars_completas].notna().all(axis=1)
cantidad = tienen_todo.sum()
porcentaje = round(cantidad / len(argentina) * 100, 1)
print("Estudiantes de Argentina con todo:", cantidad, "de", len(argentina), "(", porcentaje, "%)")

# Revisamos si los 80 pesos replicados tienen faltantes en Argentina
pesos_replicados = []
for i in range(1, 81):
    pesos_replicados.append("W_FSTURWT" + str(i))

replicados, _ = pyreadstat.read_sav(ruta_2025, usecols=["CNT"] + pesos_replicados)
replicados_arg = replicados[replicados["CNT"] == "ARG"]
con_faltantes = replicados_arg[pesos_replicados].isna().any().sum()
print("Pesos replicados con algún faltante en Argentina:", con_faltantes, "de 80")

Estudiantes de Argentina con todo: 7191 de 11092 ( 64.8 %)


Pesos replicados con algún faltante en Argentina: 0 de 80


## 6. PISA 2022 (respaldo)

Se leen solo metadatos y unas pocas columnas para comprobar qué ofrece el archivo de 2022 en Argentina y si contiene alguna pregunta de IA en las etiquetas de sus variables.

In [8]:
# Estudiantes de Argentina con dato en 2022 y variables de IA en las etiquetas
vars_2022 = ["CNT", "ESCS", "W_FSTUWT", "PV1MATH", "PV1READ", "PV1SCIE"]
datos_2022, _ = pyreadstat.read_sav(ruta_2022, usecols=vars_2022)
arg_2022 = datos_2022[datos_2022["CNT"] == "ARG"]
print("Estudiantes de Argentina en 2022:", len(arg_2022))

cobertura_2022 = arg_2022[vars_2022[1:]].notna().mean() * 100
print(cobertura_2022.round(1))

# Buscamos preguntas de IA en las etiquetas de TODAS las variables del archivo (solo metadatos)
_, meta_2022 = pyreadstat.read_sav(ruta_2022, metadataonly=True)
print("Columnas del archivo 2022:", len(meta_2022.column_names))

variables_con_ia = []
for nombre, etiqueta in meta_2022.column_names_to_labels.items():
    if etiqueta is None:
        continue
    texto = etiqueta.lower()
    if "artificial" in texto or "chatbot" in texto or "chatgpt" in texto:
        variables_con_ia.append(nombre)
print("Variables de IA en 2022:", variables_con_ia)

Estudiantes de Argentina en 2022: 12111
ESCS         96.8
W_FSTUWT    100.0
PV1MATH     100.0
PV1READ     100.0
PV1SCIE     100.0
dtype: float64
Columnas del archivo 2022: 1278
Variables de IA en 2022: []


## 7. Conclusión de viabilidad

- **Plan A (aprobado por el estudiante) es viable:** Argentina tiene valores plausibles, ESCS, pesos y preguntas de IA en el mismo archivo; la sección 5b cuantifica cuántos estudiantes tienen todo a la vez.
- **Los cuatro usos propuestos no coinciden con el instrumento.** PISA mide *frecuencia* de uso para cuatro actividades. "Copia directa" no se mide y no se puede reconstruir. Se analizará frecuencia por actividad, sin renombrarla como perfiles.
- **Riesgo principal:** alrededor de 3 de cada 10 estudiantes no tienen dato en las preguntas de IA. Hay que estudiar el origen antes de la Etapa 3.
- **Pendiente de verificar:** términos de uso de la OCDE, documentación técnica de 2025 y el factor de replicación.
- **Complementos:** PISA 2022 como contexto de rendimiento y desigualdad; UNICEF como contexto descriptivo, sin cruzarlo con PISA.

**Pendiente de revisión:** aprobar que se analizará frecuencia por actividad en lugar de los cuatro perfiles, y pasar a la Etapa 1.